# Pneumothorax Segmentation - UNet + EfficientNet-B4

Training notebook for SIIM-ACR Pneumothorax segmentation using PyTorch.

**Model:** UNet with EfficientNet-B4 encoder (ImageNet pretrained)  
**Library:** segmentation_models_pytorch  
**Dataset:** ~2500 balanced images from SIIM-ACR

## 1. Setup & Installation

In [ ]:
!git clone https://github.com/srivatsav09/pneumothorax-detection-unet.git
%cd pneumothorax-detection-unet
!git checkout pytorch-segmentation
!pip install -q segmentation-models-pytorch albumentations pydicom

## 2. Download Dataset from Kaggle

Make sure you've accepted the [competition rules](https://www.kaggle.com/c/siim-acr-pneumothorax-segmentation/rules) on Kaggle before running this.

In [ ]:
import os
from google.colab import drive

# --- Kaggle API setup ---
!pip install -q kaggle
os.environ['KAGGLE_API_TOKEN'] = '-'  # Replace with your token

# --- Download dataset (~1.2 GB compressed) ---
DATA_DIR = '/content/siim_data'
if not os.path.exists(DATA_DIR):
    !kaggle competitions download -c siim-acr-pneumothorax-segmentation -p {DATA_DIR}
    !cd {DATA_DIR} && unzip -q '*.zip'
    # Handle nested zips
    import glob
    for zf in glob.glob(f'{DATA_DIR}/**/*.zip', recursive=True):
        !unzip -qo {zf} -d {DATA_DIR}
    print('Download complete!')

# --- Locate the files ---
import glob, pathlib
csv_candidates = glob.glob(f'{DATA_DIR}/**/train-rle.csv', recursive=True)
FULL_CSV = csv_candidates[0] if csv_candidates else f'{DATA_DIR}/train-rle.csv'

dcm_files = list(pathlib.Path(DATA_DIR).rglob('*.dcm'))
FULL_IMAGE_DIR = str(dcm_files[0].parent) if dcm_files else f'{DATA_DIR}/dicom-images-train'

print(f'CSV: {FULL_CSV}')
print(f'Found {len(dcm_files)} DICOM files')

# --- Output paths ---
SUBSET_DIR = '/content/pneumothorax_subset/'
CSV_PATH = os.path.join(SUBSET_DIR, 'train-rle.csv')
IMAGE_DIR = os.path.join(SUBSET_DIR, 'images/')

# Mount Drive for saving checkpoints (persists across sessions)
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/pneumothorax_checkpoints/'
LOG_DIR = '/content/drive/MyDrive/pneumothorax_logs/'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

## 2b. Create Balanced Subset (~2500 images)

This takes all positive images (with pneumothorax) and randomly samples an equal number of negative images, then copies them to a local subset folder for fast training.

In [ ]:
import pandas as pd
import numpy as np
import shutil
import glob

np.random.seed(42)

# Read the full CSV
df = pd.read_csv(FULL_CSV)
if ' EncodedPixels' in df.columns:
    df = df.rename(columns={' EncodedPixels': 'EncodedPixels'})

print(f'Full CSV: {len(df)} rows, {df["ImageId"].nunique()} unique images')

# Classify each image as positive or negative
image_labels = {}
for img_id in df['ImageId'].unique():
    rles = df[df['ImageId'] == img_id]['EncodedPixels'].tolist()
    has_mask = any(str(r).strip() not in ('-1', 'nan', ' -1', '', '-1.0') for r in rles)
    image_labels[img_id] = has_mask

positive_ids = [k for k, v in image_labels.items() if v]
negative_ids = [k for k, v in image_labels.items() if not v]
print(f'Positive (with pneumothorax): {len(positive_ids)}')
print(f'Negative (no pneumothorax):   {len(negative_ids)}')

# Take ALL positive + equal number of random negatives
num_pos = len(positive_ids)
neg_sampled = np.random.choice(negative_ids, size=min(num_pos, len(negative_ids)), replace=False).tolist()
selected_ids = set(positive_ids + neg_sampled)
print(f'\nBalanced subset: {len(selected_ids)} images ({num_pos} pos + {len(neg_sampled)} neg)')

# Build lookup: ImageId -> actual .dcm file path (handles nested SIIM directory structure)
print('\nBuilding file index...')
dcm_lookup = {}
for dcm_path in pathlib.Path(DATA_DIR).rglob('*.dcm'):
    # The ImageId is the DICOM filename without extension
    dcm_lookup[dcm_path.stem] = str(dcm_path)
print(f'Indexed {len(dcm_lookup)} DICOM files')

# Save filtered CSV and copy files to flat subset directory
os.makedirs(IMAGE_DIR, exist_ok=True)
df_subset = df[df['ImageId'].isin(selected_ids)]
df_subset.to_csv(CSV_PATH, index=False)

copied, skipped = 0, 0
for img_id in selected_ids:
    dst = os.path.join(IMAGE_DIR, f'{img_id}.dcm')
    if os.path.exists(dst):
        continue
    src = dcm_lookup.get(img_id)
    if src:
        shutil.copy2(src, dst)
        copied += 1
    else:
        skipped += 1

existing = len([f for f in os.listdir(IMAGE_DIR) if f.endswith('.dcm')])
print(f'\nCopied {copied} files to subset folder. Skipped {skipped} (not found).')
print(f'Total in subset: {existing} DICOM files')
print(f'Subset CSV: {CSV_PATH}')

## 3. Imports & Configuration

In [ ]:
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add repo to path (adjust if needed)
REPO_DIR = '/content/pneumothorax-detection-unet'
sys.path.insert(0, REPO_DIR)

from config import DataConfig, ModelConfig, TrainConfig, AugConfig
from src.dataset import PneumothoraxDataset
from src.transforms import get_training_transforms, get_validation_transforms
from src.model import create_model, count_parameters
from src.train import Trainer, create_stratified_split, seed_everything
from src.evaluate import Evaluator, plot_training_curves

# Configuration
data_cfg = DataConfig(
    csv_path=CSV_PATH,
    image_dir=IMAGE_DIR,
    image_size=512,
)

model_cfg = ModelConfig(
    encoder_name='efficientnet-b4',
    encoder_weights='imagenet',
)

train_cfg = TrainConfig(
    batch_size=8,
    num_epochs=50,
    learning_rate=1e-4,
    use_amp=True,
    checkpoint_dir=CHECKPOINT_DIR,
    log_dir=LOG_DIR,
)

aug_cfg = AugConfig()

print('Configuration ready')

## 4. Reproducibility

In [ ]:
seed_everything(train_cfg.seed)
print(f'Seed set to {train_cfg.seed}')

## 5. Dataset & Stratified Split

In [ ]:
# Create full dataset (no transforms) for splitting
full_dataset = PneumothoraxDataset(
    csv_path=data_cfg.csv_path,
    image_dir=data_cfg.image_dir,
    image_size=data_cfg.image_size,
    rle_format='absolute',
)

# Stratified split preserving positive/negative ratio
train_idx, val_idx = create_stratified_split(
    full_dataset, train_ratio=data_cfg.train_split, seed=train_cfg.seed
)

# Create separate datasets with appropriate transforms
train_dataset = PneumothoraxDataset(
    csv_path=data_cfg.csv_path,
    image_dir=data_cfg.image_dir,
    image_size=data_cfg.image_size,
    transform=get_training_transforms(data_cfg.image_size, aug_cfg),
    rle_format='absolute',
)

val_dataset = PneumothoraxDataset(
    csv_path=data_cfg.csv_path,
    image_dir=data_cfg.image_dir,
    image_size=data_cfg.image_size,
    transform=get_validation_transforms(data_cfg.image_size),
    rle_format='absolute',
)

# Apply subset indices
train_dataset = torch.utils.data.Subset(train_dataset, train_idx)
val_dataset = torch.utils.data.Subset(val_dataset, val_idx)

print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}')

# Count positive/negative in each split
train_pos = sum(1 for i in train_idx if full_dataset.get_has_mask(i))
val_pos = sum(1 for i in val_idx if full_dataset.get_has_mask(i))
print(f'Train positive: {train_pos}/{len(train_idx)} ({train_pos/len(train_idx)*100:.1f}%)')
print(f'Val positive: {val_pos}/{len(val_idx)} ({val_pos/len(val_idx)*100:.1f}%)')

## 6. Visualize Samples (Sanity Check)

In [ ]:
imagenet_mean = np.array([0.485, 0.456, 0.406])
imagenet_std = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img, mask = train_dataset[i]

    # Denormalize for display
    img_display = img.permute(1, 2, 0).numpy()
    img_display = img_display * imagenet_std + imagenet_mean
    img_display = np.clip(img_display, 0, 1)

    axes[0, i].imshow(img_display)
    axes[0, i].set_title(f'Image {i}')
    axes[0, i].axis('off')

    axes[1, i].imshow(mask.squeeze(), cmap='Reds')
    axes[1, i].set_title(f'Mask (px={mask.sum():.0f})')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 7. Create Model

In [ ]:
model = create_model(model_cfg)
params = count_parameters(model)

print(f'Model: UNet + {model_cfg.encoder_name}')
print(f'Total params: {params["total_millions"]}M')
print(f'Trainable params: {params["trainable_millions"]}M')

# Verify forward pass
dummy = torch.randn(1, 3, 512, 512)
with torch.no_grad():
    out = model(dummy)
print(f'\nForward pass: {dummy.shape} -> {out.shape}')
assert out.shape == (1, 1, 512, 512), 'Output shape mismatch!'

## 8. Train

In [ ]:
# Check GPU
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Training will be slow.')

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    config=train_cfg,
)

trainer.train()

## 9. Training Curves

In [ ]:
curves_path = os.path.join(LOG_DIR, 'training_curves.png')
plot_training_curves(trainer.history, curves_path)

# Also display inline
from IPython.display import Image, display
display(Image(filename=curves_path))
print(f'Saved to {curves_path}')

## 10. Evaluate

In [ ]:
# Load best model
best_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pth'), weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'])
print(f'Loaded best model from epoch {best_ckpt["epoch"] + 1}')

figures_dir = os.path.join(LOG_DIR, 'figures')
evaluator = Evaluator(
    model=model,
    dataset=val_dataset,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    threshold=0.5,
)

results = evaluator.run_full_evaluation(save_dir=figures_dir)

print('\n' + '=' * 60)
print('EVALUATION RESULTS')
print('=' * 60)
metrics = results['metrics_at_default_threshold']
print(f'  Dice:              {metrics["dice_mean"]:.4f} +/- {metrics["dice_std"]:.4f}')
print(f'  IoU:               {metrics["iou_mean"]:.4f} +/- {metrics["iou_std"]:.4f}')
print(f'  Pixel Precision:   {metrics["pixel_precision"]:.4f}')
print(f'  Pixel Recall:      {metrics["pixel_recall"]:.4f}')
print(f'  Pixel F1:          {metrics["pixel_f1"]:.4f}')
print(f'  Detection Acc:     {metrics["detection_accuracy"]:.4f}')
print(f'  Detection F1:      {metrics["detection_f1"]:.4f}')
print(f'\n  Optimal Threshold: {results["optimal_threshold"]}')
print(f'\n  Positive images:   {results["positive_images"]["count"]} (Dice={results["positive_images"]["dice_mean"]:.4f})')
print(f'  Negative images:   {results["negative_images"]["count"]} (Dice={results["negative_images"]["dice_mean"]:.4f})')

## 11. Export for Deployment

In [ ]:
# Save lightweight checkpoint for Gradio (no optimizer state)
deploy_checkpoint = {
    'model_state_dict': model.state_dict(),
    'config': {
        'encoder_name': model_cfg.encoder_name,
        'image_size': data_cfg.image_size,
        'in_channels': model_cfg.in_channels,
        'classes': model_cfg.classes,
    },
    'metrics': results['metrics_at_default_threshold'],
    'optimal_threshold': results['optimal_threshold'],
}

deploy_path = os.path.join(CHECKPOINT_DIR, 'best_model_deploy.pth')
torch.save(deploy_checkpoint, deploy_path)

# Check file size
size_mb = os.path.getsize(deploy_path) / (1024 * 1024)
print(f'Deploy model saved: {deploy_path} ({size_mb:.1f} MB)')
print('\nTo deploy to HuggingFace Spaces, copy this file to app/best_model.pth')